In [ ]:
import os
import sys
from pathlib import Path

if sys.platform == 'linux':
    os.environ.setdefault('MUJOCO_GL', 'egl')
    os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')


# Shared Door + Pick-and-Place Demo

This notebook builds one simple MuJoCo scene with a real Franka arm, one hinged cabinet door, and one pickup object. It renders two scripted qualitative demonstrations in the same scene geometry:

- `open the door`
- `pick up the block and place it on the goal pedestal`

The rollouts are notebook-local demonstrations driven by IK-planned joint waypoints. They are intended to look like one coherent robot scene, not to serve as a benchmark.


## Bootstrap

Recommended environment setup from `MolmoBot/`:

```bash
uv sync --extra eval
sudo apt-get update
sudo apt-get install -y libegl1 libgl1 libgles2 libglfw3 libosmesa6 libgl1-mesa-dri mesa-utils
```

On Linux, the first code cell forces MuJoCo to use EGL for headless rendering. The notebook also expects the MolmoSpaces robot resources to be hydrated under `~/.cache/molmo-spaces-resources`.


In [ ]:
import itertools

import molmo_spaces
import mujoco
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from scipy.spatial.transform import Rotation as R
from moviepy.video.io.ImageSequenceClip import ImageSequenceClip
from IPython.display import Markdown, Video, display

from molmo_spaces.configs.robot_configs import FrankaRobotConfig
from molmo_spaces.molmo_spaces_constants import get_robot_path
from molmo_spaces.robots.franka import FrankaRobot
from molmo_spaces.robots.robot_views.franka_droid_view import FrankaDroidRobotView
from molmo_spaces.kinematics.franka_kinematics import FrankaKinematics


In [ ]:
def renderer_smoke_test():
    xml = '''
    <mujoco model="smoke">
      <worldbody>
        <geom type="plane" size="1 1 0.1"/>
        <camera name="cam" pos="1 -1 1" xyaxes="0.707 0.707 0 -0.408 0.408 0.816"/>
      </worldbody>
    </mujoco>
    '''
    model = mujoco.MjModel.from_xml_string(xml)
    data = mujoco.MjData(model)
    renderer = mujoco.Renderer(model, 120, 160)
    renderer.update_scene(data, camera='cam')
    img = renderer.render()
    renderer.close()
    return img.shape

renderer_smoke_test()


In [ ]:
OUTPUT_DIR = Path('demo_outputs/door_pick_place_shared_scene')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RENDER_HEIGHT = 360
RENDER_WIDTH = 640
VIDEO_FPS = 20

FRANKA_CFG = FrankaRobotConfig(base_size=[0.5, 0.5, 0.75])

SCENE_SPEC = {
    'object_name': 'block',
    'placement_name': 'goal pedestal',
    'door_open_angle': 1.0,
    'cabinet_pos': [-0.2, -0.1, 0.0],
    'robot_base_xy': [-0.55, -0.55],
    'robot_yaw_deg': 40.0,
    'pedestal_height': 0.74,
    'object_size': 0.04,
    'door_object_candidates': [
        [-0.20, -0.42],
        [-0.16, -0.42],
        [-0.12, -0.42],
        [-0.20, -0.38],
        [-0.16, -0.38],
    ],
    'goal_candidates': [
        [0.02, -0.26],
        [0.02, -0.22],
        [0.06, -0.26],
        [0.08, -0.22],
        [0.10, -0.26],
    ],
}

TASK_SPECS = [
    {
        'task_id': 'door_open',
        'prompt': 'open the door',
        'duration_s': 4.8,
        'manual_note': 'review_pending',
    },
    {
        'task_id': 'pick_and_place',
        'prompt': 'pick up the block and place it on the goal pedestal',
        'duration_s': 6.6,
        'manual_note': 'review_pending',
    },
]


In [ ]:
def clone_qpos_dict(qpos_dict):
    cloned = {}
    for key, value in qpos_dict.items():
        arr = np.asarray(value, dtype=float)
        cloned[key] = float(arr) if arr.ndim == 0 else arr.copy()
    return cloned


def blend_qpos_dict(q_a, q_b, alpha):
    blended = {}
    for key in q_a:
        a = np.asarray(q_a[key], dtype=float)
        b = np.asarray(q_b[key], dtype=float)
        value = (1.0 - alpha) * a + alpha * b
        blended[key] = float(value) if value.ndim == 0 else value
    return blended


def smoothstep(alpha):
    alpha = np.clip(alpha, 0.0, 1.0)
    return alpha * alpha * (3.0 - 2.0 * alpha)


def interpolate_keyframes(keyframes, phase):
    if len(keyframes) == 1:
        return clone_qpos_dict(keyframes[0]), 0, 0.0
    phase = np.clip(float(phase), 0.0, 1.0)
    scaled = phase * (len(keyframes) - 1)
    index = min(int(np.floor(scaled)), len(keyframes) - 2)
    alpha = smoothstep(scaled - index)
    return blend_qpos_dict(keyframes[index], keyframes[index + 1], alpha), index, alpha


def base_scene_path():
    return Path(molmo_spaces.__file__).resolve().parent / 'resources' / 'base_scene.xml'


def robot_quat(scene_spec):
    return R.from_euler('z', scene_spec['robot_yaw_deg'], degrees=True).as_quat(scalar_first=True).tolist()


def with_layout(scene_spec, object_xy, goal_xy):
    spec = dict(scene_spec)
    height = spec['pedestal_height']
    spec['object_xy'] = [float(object_xy[0]), float(object_xy[1])]
    spec['goal_xy'] = [float(goal_xy[0]), float(goal_xy[1])]
    spec['object_start_pos'] = [float(object_xy[0]), float(object_xy[1]), height + 0.05]
    spec['goal_object_pos'] = [float(goal_xy[0]), float(goal_xy[1]), height + 0.05]
    spec['goal_marker_pos'] = [float(goal_xy[0]), float(goal_xy[1]), height + 0.02]
    return spec


def build_scene(scene_spec):
    spec = mujoco.MjSpec.from_file(str(base_scene_path()))
    wb = spec.worldbody

    wb.add_geom(type=mujoco.mjtGeom.mjGEOM_PLANE, size=[3, 3, 0.1], pos=[0, 0, 0], rgba=[0.88, 0.87, 0.82, 1])
    wb.add_camera(name='overview', pos=[0.28, -1.46, 1.30], xyaxes=[0.97, 0.22, 0.0, -0.14, 0.59, 0.80])
    wb.add_camera(name='side', pos=[-0.62, -0.92, 1.14], xyaxes=[0.78, -0.62, 0.0, 0.36, 0.45, 0.82])

    cabinet = wb.add_body(name='cabinet_frame', pos=scene_spec['cabinet_pos'])
    for pos, size in [
        ([0.00, 0.00, 0.72], [0.03, 0.38, 0.72]),
        ([0.23, 0.00, 1.41], [0.26, 0.38, 0.03]),
        ([0.23, 0.00, 0.03], [0.26, 0.38, 0.03]),
        ([0.46, 0.00, 0.72], [0.03, 0.38, 0.72]),
        ([0.23, 0.00, 0.81], [0.20, 0.18, 0.02]),
    ]:
        cabinet.add_geom(type=mujoco.mjtGeom.mjGEOM_BOX, pos=pos, size=size, rgba=[0.95, 0.95, 0.97, 1])

    door = cabinet.add_body(name='door', pos=[0.03, -0.38, 0.72])
    door.add_joint(name='door_hinge', type=mujoco.mjtJoint.mjJNT_HINGE, axis=[0, 0, 1], range=[0, 1.2], damping=0.2)
    door.add_geom(type=mujoco.mjtGeom.mjGEOM_BOX, pos=[0.20, 0.02, 0.0], size=[0.20, 0.02, 0.64], rgba=[0.58, 0.38, 0.20, 1])
    door.add_geom(type=mujoco.mjtGeom.mjGEOM_CAPSULE, fromto=[0.31, 0.035, -0.04, 0.37, 0.035, 0.04], size=[0.016], rgba=[0.95, 0.82, 0.32, 1])
    door.add_site(name='handle_site', pos=[0.37, 0.035, 0.0], size=[0.014], rgba=[1.0, 0.2, 0.2, 1.0])

    height = scene_spec['pedestal_height']
    object_xy = np.asarray(scene_spec['object_xy'], dtype=float)
    goal_xy = np.asarray(scene_spec['goal_xy'], dtype=float)
    wb.add_body(name='object_pedestal', pos=[float(object_xy[0]), float(object_xy[1]), height / 2.0]).add_geom(
        type=mujoco.mjtGeom.mjGEOM_CYLINDER,
        size=[0.06, height / 2.0],
        rgba=[0.72, 0.72, 0.76, 1.0],
    )
    wb.add_body(name='goal_pedestal', pos=[float(goal_xy[0]), float(goal_xy[1]), height / 2.0]).add_geom(
        type=mujoco.mjtGeom.mjGEOM_CYLINDER,
        size=[0.07, height / 2.0],
        rgba=[0.82, 0.70, 0.63, 1.0],
    )

    pickup = wb.add_body(name='pickup_object', pos=scene_spec['object_start_pos'])
    pickup.add_freejoint(name='object_free')
    pickup.add_geom(
        type=mujoco.mjtGeom.mjGEOM_BOX,
        size=[scene_spec['object_size'], scene_spec['object_size'], scene_spec['object_size']],
        rgba=[0.14, 0.42, 0.80, 1.0],
    )
    pickup.add_site(name='object_site', pos=[0, 0, 0], size=[0.012], rgba=[0.15, 0.55, 1.0, 1.0])

    goal = wb.add_body(name='goal_marker', pos=scene_spec['goal_marker_pos'])
    goal.add_geom(type=mujoco.mjtGeom.mjGEOM_CYLINDER, size=[0.075, 0.012], rgba=[0.80, 0.20, 0.18, 0.72], contype=0, conaffinity=0)

    robot_spec = mujoco.MjSpec.from_file(str(get_robot_path(FRANKA_CFG.name) / FRANKA_CFG.robot_xml_path))
    FrankaRobot.add_robot_to_scene(
        FRANKA_CFG,
        spec,
        robot_spec,
        prefix=FRANKA_CFG.robot_namespace,
        pos=scene_spec['robot_base_xy'],
        quat=robot_quat(scene_spec),
    )

    model = spec.compile()
    data = mujoco.MjData(model)
    renderer = mujoco.Renderer(model, RENDER_HEIGHT, RENDER_WIDTH)
    view = FrankaDroidRobotView(data, FRANKA_CFG.robot_namespace)
    view.set_qpos_dict(FRANKA_CFG.init_qpos)
    mujoco.mj_forward(model, data)
    kin = FrankaKinematics(model, data, namespace=FRANKA_CFG.robot_namespace, robot_view_factory=FrankaDroidRobotView)

    object_joint_id = model.joint('object_free').id
    object_qpos_adr = model.jnt_qposadr[object_joint_id]
    door_joint_id = model.joint('door_hinge').id
    door_qpos_adr = model.jnt_qposadr[door_joint_id]

    return {
        'model': model,
        'data': data,
        'renderer': renderer,
        'view': view,
        'kin': kin,
        'scene_spec': scene_spec,
        'object_qpos_adr': object_qpos_adr,
        'door_qpos_adr': door_qpos_adr,
        'camera_names': ['overview', 'side'],
    }


def close_scene(scene_ctx):
    scene_ctx['renderer'].close()


def refresh_scene(scene_ctx):
    mujoco.mj_forward(scene_ctx['model'], scene_ctx['data'])


def set_qpos_dict(scene_ctx, qpos_dict):
    scene_ctx['view'].set_qpos_dict(qpos_dict)
    refresh_scene(scene_ctx)


def set_door_angle(scene_ctx, angle):
    scene_ctx['data'].qpos[scene_ctx['door_qpos_adr']] = angle
    refresh_scene(scene_ctx)


def set_object_pose(scene_ctx, pos, quat=(1.0, 0.0, 0.0, 0.0)):
    adr = scene_ctx['object_qpos_adr']
    scene_ctx['data'].qpos[adr:adr + 3] = np.asarray(pos, dtype=float)
    scene_ctx['data'].qpos[adr + 3:adr + 7] = np.asarray(quat, dtype=float)
    refresh_scene(scene_ctx)


def ee_pose(scene_ctx):
    return scene_ctx['view'].get_move_group('arm').leaf_frame_to_world.copy()


def site_pos(scene_ctx, site_name):
    site_id = scene_ctx['model'].site(site_name).id
    return scene_ctx['data'].site_xpos[site_id].copy()


def render_scene(scene_ctx, camera_names=None):
    if camera_names is None:
        camera_names = scene_ctx['camera_names']
    images = {}
    for camera_name in camera_names:
        scene_ctx['renderer'].update_scene(scene_ctx['data'], camera=camera_name)
        images[camera_name] = scene_ctx['renderer'].render().copy()
    return images


def preview_scene(scene_spec):
    scene_ctx = build_scene(scene_spec)
    images = render_scene(scene_ctx)
    close_scene(scene_ctx)
    return Image.fromarray(np.hstack([images['overview'], images['side']]))


In [ ]:
def solve_pose_waypoints(scene_ctx, targets, rotation, q_start, max_iter=700):
    q_current = clone_qpos_dict(q_start)
    solved = []
    base_pose = scene_ctx['view'].base.pose.copy()
    for target in targets:
        pose = np.eye(4)
        pose[:3, :3] = rotation
        pose[:3, 3] = np.asarray(target, dtype=float)
        q_next = scene_ctx['kin'].ik('arm', pose, ['arm'], q_current, base_pose, max_iter=max_iter, dt=0.25)
        if q_next is None:
            return None
        q_current = clone_qpos_dict(q_next)
        solved.append(q_current)
    return solved


def handle_target(scene_ctx, door_angle, offset):
    set_door_angle(scene_ctx, door_angle)
    return site_pos(scene_ctx, 'handle_site') + np.asarray(offset, dtype=float)


def plan_door_task(scene_spec):
    scene_ctx = build_scene(scene_spec)
    try:
        q_home = clone_qpos_dict(scene_ctx['view'].get_qpos_dict())
        ee_rot = ee_pose(scene_ctx)[:3, :3].copy()
        targets = [
            handle_target(scene_ctx, 0.0, [-0.12, -0.08, 0.04]),
            handle_target(scene_ctx, 0.0, [-0.04, -0.02, 0.00]),
            handle_target(scene_ctx, 0.35, [-0.03, -0.02, 0.00]),
            handle_target(scene_ctx, 0.70, [-0.03, -0.02, 0.00]),
            handle_target(scene_ctx, scene_spec['door_open_angle'], [-0.03, -0.02, 0.00]),
            handle_target(scene_ctx, scene_spec['door_open_angle'], [-0.18, -0.10, 0.10]),
        ]
        solved = solve_pose_waypoints(scene_ctx, targets, ee_rot, q_home)
        if solved is None:
            raise RuntimeError('Door-open IK planning failed.')
        return {
            'keyframes': [q_home] + solved,
            'door_angles': [0.0, 0.0, 0.0, 0.35, 0.70, scene_spec['door_open_angle'], scene_spec['door_open_angle']],
            'object_mode': 'static',
        }
    finally:
        close_scene(scene_ctx)


def pick_rotation_candidates(init_rot):
    return [
        ('init', init_rot),
        ('x180', R.from_euler('x', 180, degrees=True).as_matrix() @ init_rot),
        ('y180', R.from_euler('y', 180, degrees=True).as_matrix() @ init_rot),
        ('z90', R.from_euler('z', 90, degrees=True).as_matrix() @ init_rot),
        ('xz', R.from_euler('xz', [180, 90], degrees=True).as_matrix() @ init_rot),
    ]


def try_pick_task(scene_spec):
    scene_ctx = build_scene(scene_spec)
    try:
        q_home = clone_qpos_dict(scene_ctx['view'].get_qpos_dict())
        init_rot = ee_pose(scene_ctx)[:3, :3].copy()
        object_pos = np.asarray(scene_spec['object_start_pos'], dtype=float)
        goal_pos = np.asarray(scene_spec['goal_object_pos'], dtype=float)
        targets = [
            object_pos + np.array([0.0, 0.0, 0.14]),
            object_pos + np.array([0.0, 0.0, 0.05]),
            object_pos + np.array([0.0, 0.0, 0.05]),
            object_pos + np.array([0.0, 0.0, 0.20]),
            0.5 * (object_pos + goal_pos) + np.array([0.0, 0.0, 0.24]),
            goal_pos + np.array([0.0, 0.0, 0.18]),
            goal_pos + np.array([0.0, 0.0, 0.07]),
            goal_pos + np.array([0.0, 0.0, 0.07]),
            goal_pos + np.array([0.0, 0.0, 0.18]),
        ]
        for label, rotation in pick_rotation_candidates(init_rot):
            solved = solve_pose_waypoints(scene_ctx, targets, rotation, q_home, max_iter=300)
            if solved is None:
                continue
            set_qpos_dict(scene_ctx, solved[2])
            grasp_ee = ee_pose(scene_ctx)[:3, 3].copy()
            attach_offset = object_pos - grasp_ee
            return {
                'rotation_label': label,
                'keyframes': [q_home] + solved,
                'door_angles': [0.0] * (len(solved) + 1),
                'object_mode': 'carry',
                'attach_offset': attach_offset,
                'carry_start_index': 2,
                'carry_end_index': 8,
            }
        return None
    finally:
        close_scene(scene_ctx)


def prepare_demo_plan(scene_spec=SCENE_SPEC):
    door_scene_spec = with_layout(scene_spec, scene_spec['door_object_candidates'][0], scene_spec['goal_candidates'][0])
    door_task = plan_door_task(door_scene_spec)

    for object_xy, goal_xy in itertools.product(scene_spec['door_object_candidates'], scene_spec['goal_candidates']):
        candidate = with_layout(scene_spec, object_xy, goal_xy)
        pick_task = try_pick_task(candidate)
        if pick_task is not None:
            return {
                'scene_spec': candidate,
                'tasks': {
                    'door_open': door_task,
                    'pick_and_place': pick_task,
                },
            }
    raise RuntimeError('Failed to find a reachable pick-and-place layout for the Franka demo scene.')


DEMO_PLAN = prepare_demo_plan()
ACTIVE_SCENE_SPEC = DEMO_PLAN['scene_spec']
ACTIVE_SCENE_SPEC


## Preview The Shared Scene


In [ ]:
preview_scene(ACTIVE_SCENE_SPEC)


## Run The Demo Tasks

Each task starts from the same real-robot scene. The door-open sequence uses handle-locked IK keyframes, and the pick-and-place sequence moves the block between two nearby pedestals while keeping the same cabinet and robot in frame.


In [ ]:
def apply_rollout_state(scene_ctx, task_plan, phase):
    qpos_dict, index, alpha = interpolate_keyframes(task_plan['keyframes'], phase)
    door_angle = (1.0 - alpha) * task_plan['door_angles'][index] + alpha * task_plan['door_angles'][index + 1]
    set_qpos_dict(scene_ctx, qpos_dict)
    set_door_angle(scene_ctx, door_angle)

    if task_plan['object_mode'] == 'static':
        set_object_pose(scene_ctx, scene_ctx['scene_spec']['object_start_pos'])
        return

    object_start = np.asarray(scene_ctx['scene_spec']['object_start_pos'], dtype=float)
    object_goal = np.asarray(scene_ctx['scene_spec']['goal_object_pos'], dtype=float)
    ee_pos = ee_pose(scene_ctx)[:3, 3]
    carry_start = task_plan.get('carry_start_index', 2)
    carry_end = task_plan.get('carry_end_index', len(task_plan['keyframes']) - 2)
    if index < carry_start:
        object_pos = object_start
    elif index < carry_end:
        object_pos = ee_pos + np.asarray(task_plan['attach_offset'], dtype=float)
    else:
        object_pos = object_goal
    set_object_pose(scene_ctx, object_pos)


def run_rollout(task_spec, demo_plan=DEMO_PLAN, output_dir=OUTPUT_DIR, fps=VIDEO_FPS):
    scene_ctx = build_scene(demo_plan['scene_spec'])
    n_steps = max(2, round(task_spec['duration_s'] * fps))
    frames = []
    task_plan = demo_plan['tasks'][task_spec['task_id']]

    for step in tqdm(range(n_steps), desc=task_spec['task_id']):
        phase = step / (n_steps - 1)
        apply_rollout_state(scene_ctx, task_plan, phase)
        images = render_scene(scene_ctx)
        frames.append(np.hstack([images['overview'], images['side']]))

    video_path = output_dir / f"{task_spec['task_id']}.mp4"
    clip = ImageSequenceClip([frame for frame in frames], fps=fps)
    clip.write_videofile(str(video_path), audio=False, logger=None)
    clip.close()
    close_scene(scene_ctx)

    return {
        'task_id': task_spec['task_id'],
        'prompt': task_spec['prompt'],
        'scene': 'shared_real_franka_door_pick_place_scene',
        'video_path': str(video_path),
        'manual_note': task_spec['manual_note'],
    }


def run_task_suite(task_specs=TASK_SPECS):
    return pd.DataFrame([run_rollout(task_spec) for task_spec in task_specs])


results_df = run_task_suite()
results_df


In [ ]:
for record in results_df.to_dict(orient='records'):
    summary = (
        f"### {record['task_id']}\n"
        f"- prompt: `{record['prompt']}`\n"
        f"- scene: `{record['scene']}`\n"
        f"- note: `{record['manual_note']}`"
    )
    display(Markdown(summary))
    display(Video(record['video_path'], embed=True))
